# 🎨 ComfyUI trên Google Colab (Free) — bản lưu model vào Google Drive

**Cách dùng:**
1. `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save
2. Chạy lần lượt **Cell 1 → 2 → 3**, copy link ở Cell 3 dán vào tab mới
3. Cell 4 = kiểm tra sức khỏe; Cell 5 = link dự phòng (localtunnel)

**💾 Điểm mới:** model lưu trong Google Drive của bạn → khởi động lại KHÔNG phải tải lại model (tiết kiệm 5-10 phút mỗi lần).
- Lần đầu: ~10 phút (tải model 1 lần duy nhất)
- Các lần sau: chỉ ~3 phút (cài code + nạp model từ Drive)

⚠️ Cần Google Drive còn trống ~10GB (Animagine 7GB + SD1.5 2GB). Xóa bớt model trong thư mục `AI_Models` trên Drive nếu thiếu chỗ.

In [ ]:
# ===== CELL 1: Kết nối Google Drive + Cài ComfyUI (~2-3 phút) =====
!nvidia-smi --query-gpu=name,memory.total --format=csv

# 1) Kết nối Google Drive (hiện cửa sổ xin quyền -> chọn tài khoản -> Cho phép)
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục chứa model trên Drive (tồn tại vĩnh viễn giữa các phiên)
import os
os.makedirs('/content/drive/MyDrive/AI_Models/checkpoints', exist_ok=True)

# 2) Cài ComfyUI (bắt buộc mỗi phiên - phần này nhanh)
os.chdir('/content')
!rm -rf /content/ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
os.chdir('/content/ComfyUI')
!pip install -q -r requirements.txt

# 3) Trỏ thư mục model của ComfyUI sang Drive (symlink)
!rm -rf /content/ComfyUI/models/checkpoints
!ln -s /content/drive/MyDrive/AI_Models/checkpoints /content/ComfyUI/models/checkpoints

print('\n✅ Xong Cell 1! Model có sẵn trong Drive:')
!ls -lh /content/drive/MyDrive/AI_Models/checkpoints/ 2>/dev/null || echo '(chưa có model nào - Cell 2 sẽ tải)'

In [ ]:
# ===== CELL 2: Tải Model vào Drive (CHỈ TẢI LẦN ĐẦU, lần sau tự bỏ qua) =====
# wget -c: nếu file đã có đủ trong Drive thì bỏ qua ngay, không tải lại

# Model anime chất lượng cao Animagine XL 4.0 (6.94 GB)
!wget -c -O /content/drive/MyDrive/AI_Models/checkpoints/animagine-xl-4.0-opt.safetensors \
  "https://huggingface.co/cagliostrolab/animagine-xl-4.0/resolve/main/animagine-xl-4.0-opt.safetensors"

# (TÙY CHỌN) SD 1.5 nhẹ + nhanh (2.13 GB) - bỏ dấu # nếu muốn
#!wget -c -O /content/drive/MyDrive/AI_Models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors \
#  "https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors"

!ls -lh /content/drive/MyDrive/AI_Models/checkpoints/
print('\n✅ Model sẵn sàng (đã nằm trong Drive, lần sau không phải tải lại)!')

In [ ]:
# ===== CELL 3: Khởi chạy ComfyUI (chạy NỀN) + tạo link truy cập =====
import subprocess, time, socket, re, os

# Dọn tiến trình cũ nếu chạy lại cell này
!pkill -f "python main.py" 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

# Cài cloudflared (nếu chưa có)
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 1) Chạy ComfyUI NỀN
os.chdir('/content/ComfyUI')
comfy_log = open('/content/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--enable-cors-header'],
    stdout=comfy_log, stderr=subprocess.STDOUT)
print('⏳ Đang khởi động ComfyUI (30-60 giây)...')

# 2) Đợi cổng 8188 mở
for _ in range(180):
    time.sleep(1)
    if comfy.poll() is not None:
        raise RuntimeError('❌ ComfyUI bị tắt! Xem lỗi bằng: !tail -30 /content/comfyui.log')
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        pass
else:
    raise RuntimeError('❌ Quá 3 phút chưa mở cổng. Xem log: !tail -30 /content/comfyui.log')
print('✅ ComfyUI đã chạy!')

# 3) Chạy cloudflared NỀN (http2 - hợp mạng VN)
cf_log = open('/content/cloudflared.log', 'w')
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188',
     '--http-host-header', '127.0.0.1:8188', '--protocol', 'http2'],
    stdout=cf_log, stderr=subprocess.STDOUT)

# 4) Đọc link
url = None
for _ in range(60):
    time.sleep(1)
    txt = open('/content/cloudflared.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        break
print()
print('='*60)
if url:
    print('🎨 COPY LINK NÀY, DÁN VÀO THANH ĐỊA CHỈ TAB MỚI:')
    print(url)
else:
    print('⚠️ Chưa lấy được link cloudflare — dùng Cell 5 (localtunnel)')
print('='*60)
print('\n💡 Cell này đã xong nhưng ComfyUI vẫn chạy nền.')

In [ ]:
# ===== CELL 4: KIỂM TRA sức khỏe hệ thống (chạy bất cứ lúc nào) =====
!curl -s -o /dev/null -w "A) ComfyUI noi bo:  HTTP %{http_code} (200 = OK)\n" --max-time 20 http://127.0.0.1:8188/system_stats

import re
txt = open('/content/cloudflared.log').read()
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
if m:
    url = m.group(0)
    print('   Link hien tai:', url)
    !curl -s -o /dev/null -w "B) Qua tunnel:      HTTP %{http_code} (200 = OK, 502 = ComfyUI chet, 000 = tunnel chet)\n" --max-time 40 {url}/system_stats
else:
    print('B) Khong tim thay link trong log cloudflared')

print('\n----- LOG ComfyUI (30 dòng cuối) -----')
!tail -30 /content/comfyui.log

In [ ]:
# ===== CELL 5 (DỰ PHÒNG): Link thay thế qua localtunnel =====
!npm install -g localtunnel > /dev/null 2>&1

import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('='*60)
print('🔑 MẬT KHẨU (Tunnel Password) khi trang hỏi:', ip)
print('='*60)
print('Đợi link https://....loca.lt hiện ra bên dưới rồi mở link đó.\n')

!lt --port 8188

## 📝 Ghi chú

**Khởi động lại phiên mới chỉ cần:** Cell 1 → (Cell 2 tự bỏ qua nếu model đã có) → Cell 3. Tổng ~3 phút.

**Workflow tiếng Việt cho Animagine XL:** tải file `workflow_animagine_xl_tiengviet.json` rồi kéo thả vào ComfyUI.

**Ảnh đã tạo cũng có thể lưu vào Drive:** ảnh nằm ở `/content/ComfyUI/output` — muốn tự động lưu vĩnh viễn, chạy thêm:
```python
!rm -rf /content/ComfyUI/output
!mkdir -p /content/drive/MyDrive/AI_Models/output
!ln -s /content/drive/MyDrive/AI_Models/output /content/ComfyUI/output
```
(chạy TRƯỚC Cell 3, ảnh sẽ nằm trong Drive → thư mục `AI_Models/output`)

**Xử lý sự cố:**
- Không thấy GPU → Runtime → Change runtime type → T4 GPU
- Link treo/403 → copy link dán vào thanh địa chỉ tab MỚI (đừng bấm trực tiếp)
- 502 → ComfyUI chết, chạy Cell 4 xem log rồi chạy lại Cell 3
- Drive hết chỗ → xóa bớt file trong `MyDrive/AI_Models/checkpoints`